# 🥔 Potato Price Prediction using Machine Learning
## Assignment 2 — Agricultural Commodity Price Forecasting

**Objective:** Predict potato mandi (market) prices using historical price trends, seasonal patterns and weather variables, then expose the collected data through natural-language prompts, a retrieval-augmented chat assistant, and a dashboard.

### Important data note
The supplied repository already contains the prepared `Potato_Price_TimeSeries_Weather_Dataset.csv/xlsx`. The original raw snapshot used to construct this file is **not included in this repository**, so this final notebook does **not** depend on a missing raw file. The README documents that the 52-week history and weather fields are a simulation anchored to real market snapshot prices. Results must therefore be described as a project/academic forecasting experiment, not as evidence of production accuracy on real multi-year historical data.

### Pipeline
1. Load and validate the prepared dataset
2. Explore price, season and weather patterns
3. Create lag/rolling time-series features
4. Compare regression models
5. Evaluate varied Random Forest hyperparameters
6. Tune the model with GridSearchCV
7. Demonstrate documented natural-language data extraction prompts
8. Build a context-aware TF-IDF retrieval + response generation RAG-style assistant
9. Export dashboard-ready results
10. Summarize limitations and next steps

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
np.random.seed(42)

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data", "Potato_Price_TimeSeries_Weather_Dataset.csv")
os.makedirs(os.path.join(ROOT, "outputs"), exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["Arrival_Date"])
print("Dataset shape:", df.shape)
print("Date range:", df["Arrival_Date"].min().date(), "to", df["Arrival_Date"].max().date())
print("States:", df["State"].nunique(), "| Markets:", df["Market"].nunique())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
df.head()

## 1. Dataset validation

The model target is `Modal_Price_Rs_per_Quintal`. The dataset contains weekly market observations with state, district, market, season, weather and price fields.

In [ ]:
required_cols = [
    "State","District","Market","Commodity","Variety","Grade","Arrival_Date",
    "Year","Month","Week","Season","Rainfall_mm","Temperature_C","Humidity_percent",
    "Min_Price_Rs_per_Quintal","Max_Price_Rs_per_Quintal","Modal_Price_Rs_per_Quintal"
]
missing_required = [c for c in required_cols if c not in df.columns]
assert not missing_required, f"Missing required columns: {missing_required}"
assert (df["Modal_Price_Rs_per_Quintal"] > 0).all()
print("Validation passed.")
print(df[required_cols].describe(include="all").T.head(20))

## 2. Exploratory Data Analysis

In [ ]:
season_order = ["Winter","Summer","Monsoon","Post-Monsoon"]
season_avg = df.groupby("Season")["Modal_Price_Rs_per_Quintal"].mean().reindex(season_order)

plt.figure(figsize=(8,4))
season_avg.plot(kind="bar")
plt.ylabel("Rs / quintal")
plt.title("Average Potato Price by Season")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

weekly = df.groupby("Arrival_Date", as_index=False)["Modal_Price_Rs_per_Quintal"].mean()
weekly["Moving_Avg_8wk"] = weekly["Modal_Price_Rs_per_Quintal"].rolling(8).mean()

plt.figure(figsize=(11,4))
plt.plot(weekly["Arrival_Date"], weekly["Modal_Price_Rs_per_Quintal"], alpha=.45, label="Weekly average")
plt.plot(weekly["Arrival_Date"], weekly["Moving_Avg_8wk"], linewidth=2, label="8-week moving average")
plt.title("National Weekly Potato Price Trend")
plt.ylabel("Rs / quintal")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

weather_corr = df[["Rainfall_mm","Temperature_C","Humidity_percent","Modal_Price_Rs_per_Quintal"]].corr()
plt.figure(figsize=(6,4.5))
sns.heatmap(weather_corr, annot=True, fmt=".2f")
plt.title("Weather / Price Correlation")
plt.tight_layout()
plt.show()

## 3. Time-series feature engineering

For each market, create:
- `Lag1`: previous week's modal price
- `Lag2`: price two weeks earlier
- `Rolling_Mean_4wk`: previous four-week average

A **chronological holdout** is used: the latest 8 weeks are the test period. This avoids randomly mixing future observations into the training set.

In [ ]:
df = df.sort_values(["State","District","Market","Arrival_Date"]).reset_index(drop=True)
group_cols = ["State","District","Market"]

df["Modal_Price_Lag1"] = df.groupby(group_cols)["Modal_Price_Rs_per_Quintal"].shift(1)
df["Modal_Price_Lag2"] = df.groupby(group_cols)["Modal_Price_Rs_per_Quintal"].shift(2)
df["Rolling_Mean_4wk"] = (
    df.groupby(group_cols)["Modal_Price_Rs_per_Quintal"]
      .transform(lambda s: s.shift(1).rolling(4).mean())
)

model_df = df.dropna(subset=["Modal_Price_Lag1","Modal_Price_Lag2","Rolling_Mean_4wk"]).copy()
cutoff = model_df["Arrival_Date"].max() - pd.Timedelta(weeks=8)

train_mask = model_df["Arrival_Date"] <= cutoff
test_mask = model_df["Arrival_Date"] > cutoff

y = model_df["Modal_Price_Rs_per_Quintal"]

cat_cols = ["State","Grade","Season"]
weather_cols = ["Rainfall_mm","Temperature_C","Humidity_percent","Month"]
lag_cols = ["Modal_Price_Lag1","Modal_Price_Lag2","Rolling_Mean_4wk"]

X_no_lag = pd.get_dummies(model_df[cat_cols + weather_cols], columns=cat_cols, drop_first=True)
X_lag = pd.get_dummies(model_df[cat_cols + weather_cols + lag_cols], columns=cat_cols, drop_first=True)

print("Train:", train_mask.sum(), "rows")
print("Test :", test_mask.sum(), "rows")
print("Test period:", model_df.loc[test_mask, "Arrival_Date"].min().date(), "to", model_df.loc[test_mask, "Arrival_Date"].max().date())

## 4. Regression model comparison

In [ ]:
def metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred)
    }

results = {}

rf_no_lag = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_no_lag.fit(X_no_lag.loc[train_mask], y.loc[train_mask])
results["Random Forest - no price history"] = metrics(
    y.loc[test_mask], rf_no_lag.predict(X_no_lag.loc[test_mask])
)

rf_lag = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_lag.fit(X_lag.loc[train_mask], y.loc[train_mask])
results["Random Forest - with price history"] = metrics(
    y.loc[test_mask], rf_lag.predict(X_lag.loc[test_mask])
)

simple = LinearRegression().fit(model_df.loc[train_mask, ["Modal_Price_Lag1"]], y.loc[train_mask])
results["Simple Linear Regression"] = metrics(
    y.loc[test_mask], simple.predict(model_df.loc[test_mask, ["Modal_Price_Lag1"]])
)

multi_features = ["Modal_Price_Lag1","Modal_Price_Lag2","Rolling_Mean_4wk"]
multi = LinearRegression().fit(model_df.loc[train_mask, multi_features], y.loc[train_mask])
results["Multiple Linear Regression"] = metrics(
    y.loc[test_mask], multi.predict(model_df.loc[test_mask, multi_features])
)

poly = make_pipeline(PolynomialFeatures(2), LinearRegression())
poly.fit(model_df.loc[train_mask, ["Modal_Price_Lag1"]], y.loc[train_mask])
results["Polynomial Regression (degree 2)"] = metrics(
    y.loc[test_mask], poly.predict(model_df.loc[test_mask, ["Modal_Price_Lag1"]])
)

ridge = Ridge(alpha=1.0)
ridge.fit(model_df.loc[train_mask, multi_features], y.loc[train_mask])
results["Ridge Regression"] = metrics(
    y.loc[test_mask], ridge.predict(model_df.loc[test_mask, multi_features])
)

results_df = pd.DataFrame(results).T.round(4)
results_df

### Interpreting the comparison

The large improvement after adding lag features shows that recent price history carries strong information about the next week's price in this constructed dataset. The high R² values should **not** be presented as production-level accuracy because the underlying historical/weather series is simulated and highly autocorrelated.

## 5. Hyperparameter experiments

In [ ]:
hp_rows = []
for n_estimators in [10, 50, 100, 200]:
    for max_depth in [3, 5, 10, None]:
        model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            random_state=42, n_jobs=-1
        )
        model.fit(X_lag.loc[train_mask], y.loc[train_mask])
        pred = model.predict(X_lag.loc[test_mask])
        row = metrics(y.loc[test_mask], pred)
        row.update({"n_estimators": n_estimators, "max_depth": "None" if max_depth is None else max_depth})
        hp_rows.append(row)

hp_df = pd.DataFrame(hp_rows)[["n_estimators","max_depth","MAE","RMSE","R2"]]
display(hp_df.sort_values("RMSE").round(4))

plt.figure(figsize=(8,5))
pivot = hp_df.pivot(index="max_depth", columns="n_estimators", values="R2")
sns.heatmap(pivot, annot=True, fmt=".4f")
plt.title("Random Forest R² across Hyperparameters")
plt.tight_layout()
plt.show()

## 6. Time-series cross-validation and GridSearchCV

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
cv_scores = []
for train_idx, val_idx in tscv.split(X_lag):
    cv_model.fit(X_lag.iloc[train_idx], y.iloc[train_idx])
    cv_scores.append(r2_score(y.iloc[val_idx], cv_model.predict(X_lag.iloc[val_idx])))

print("TimeSeriesSplit R²:", np.round(cv_scores, 4))
print("Mean CV R²:", round(float(np.mean(cv_scores)), 4))
print("Std CV R² :", round(float(np.std(cv_scores)), 4))

grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {
        "n_estimators": [100, 200],
        "max_depth": [5, 10],
        "min_samples_split": [2, 5]
    },
    cv=tscv,
    scoring="r2",
    n_jobs=-1
)
grid.fit(X_lag.loc[train_mask], y.loc[train_mask])

best_model = grid.best_estimator_
tuned_pred = best_model.predict(X_lag.loc[test_mask])
tuned_metrics = metrics(y.loc[test_mask], tuned_pred)

print("Best parameters:", grid.best_params_)
print("Best CV R²:", round(grid.best_score_, 4))
print("Held-out test metrics:", {k: round(v,4) for k,v in tuned_metrics.items()})

## 7. Natural-language prompt-based data extraction

The following prompts are explicitly documented for the assignment. They are **query prompts**, not claims that an LLM generated the underlying dataset.

### Prompts used
1. “What is the current potato price in Tamil Nadu?”
2. “What is the average potato price in Punjab during monsoon?”
3. “How much does potato cost during winter?”
4. “Which states have the highest potato prices?”
5. “Is the potato price trend rising or falling?”
6. “What is the price in Uttar Pradesh in post-monsoon season?”

The parser extracts entities from the prompt and retrieves the corresponding records.

In [ ]:
latest_date = df["Arrival_Date"].max()
latest = df[df["Arrival_Date"] == latest_date]
states = sorted(df["State"].dropna().unique())
seasons = ["Post-Monsoon","Winter","Summer","Monsoon"]

def parse_entities(question, previous_context=None):
    q = question.lower()
    state = next((s for s in states if s.lower() in q), None)
    season = next((s for s in seasons if s.lower() in q), None)
    if state is None and previous_context:
        state = previous_context.get("state")
    if season is None and previous_context:
        season = previous_context.get("season")
    return state, season

def prompt_extract(question, previous_context=None):
    state, season = parse_entities(question, previous_context)
    q = question.lower()

    if state and season:
        sub = df[(df["State"] == state) & (df["Season"] == season)]
        if len(sub):
            return {
                "type":"state_season", "state":state, "season":season,
                "records":len(sub), "avg_price":float(sub["Modal_Price_Rs_per_Quintal"].mean())
            }

    if state:
        sub = latest[latest["State"] == state]
        if len(sub):
            return {
                "type":"state_current", "state":state, "season":season,
                "records":len(sub), "avg_price":float(sub["Modal_Price_Rs_per_Quintal"].mean())
            }

    if season:
        sub = df[df["Season"] == season]
        return {
            "type":"season", "state":state, "season":season,
            "records":len(sub), "avg_price":float(sub["Modal_Price_Rs_per_Quintal"].mean())
        }

    if "highest" in q or "top" in q:
        top = latest.groupby("State")["Modal_Price_Rs_per_Quintal"].mean().sort_values(ascending=False).head(5)
        return {"type":"top_states", "top":top.to_dict()}

    if "trend" in q or "rising" in q or "falling" in q:
        first = float(df[df["Arrival_Date"] == df["Arrival_Date"].min()]["Modal_Price_Rs_per_Quintal"].mean())
        last = float(latest["Modal_Price_Rs_per_Quintal"].mean())
        return {"type":"trend", "first":first, "last":last, "direction":"rising" if last >= first else "falling"}

    return {"type":"overall", "avg_price":float(latest["Modal_Price_Rs_per_Quintal"].mean())}

for prompt in [
    "What is the current potato price in Tamil Nadu?",
    "What is the average potato price in Punjab during monsoon?",
    "How much does potato cost during winter?",
    "Which states have the highest potato prices?",
    "Is the potato price trend rising or falling?",
    "What is the price in Uttar Pradesh in post-monsoon season?"
]:
    print("PROMPT:", prompt)
    print("EXTRACTED:", prompt_extract(prompt))
    print()

## 8. RAG-style chat assistant with conversational context

This implementation has three explicit stages:

**Retrieve → Augment with context → Generate**

- **Retrieve:** TF-IDF retrieves the most relevant state/season/month/market summary documents from the collected dataset.
- **Context:** the previous turn's state/season is retained when a follow-up says “there”, “that state”, or omits the entity.
- **Generate:** a deterministic natural-language response is produced from the retrieved numeric facts. No external LLM/API is required.

This is a lightweight academic RAG implementation; a production system could replace the deterministic generator with an LLM while keeping the retrieval layer.

In [ ]:
# Build compact retrieval documents from the collected data.
rag_docs = []

for (state, season), g in df.groupby(["State","Season"]):
    rag_docs.append({
        "text": f"Potato price in {state} during {season}. Average modal price is {g['Modal_Price_Rs_per_Quintal'].mean():.0f} rupees per quintal. "
                f"Rainfall average {g['Rainfall_mm'].mean():.1f} mm, temperature {g['Temperature_C'].mean():.1f} C, humidity {g['Humidity_percent'].mean():.1f} percent.",
        "state": state, "season": season, "kind": "state_season",
        "avg_price": float(g["Modal_Price_Rs_per_Quintal"].mean())
    })

for month, g in df.groupby("Month"):
    rag_docs.append({
        "text": f"National potato price for month {month}. Average modal price is {g['Modal_Price_Rs_per_Quintal'].mean():.0f} rupees per quintal.",
        "state": None, "season": None, "kind": "month",
        "avg_price": float(g["Modal_Price_Rs_per_Quintal"].mean())
    })

rag_texts = [d["text"] for d in rag_docs]
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(rag_texts)

chat_context = {"state": None, "season": None}

def retrieve(question, top_k=3):
    qv = vectorizer.transform([question])
    scores = cosine_similarity(qv, doc_matrix).ravel()
    idx = np.argsort(scores)[::-1][:top_k]
    return [(rag_docs[i], float(scores[i])) for i in idx]

def rag_answer(question):
    global chat_context
    explicit_state, explicit_season = parse_entities(question, chat_context)
    q = question.lower()

    # Preserve conversation context for follow-up questions.
    if explicit_state:
        chat_context["state"] = explicit_state
    if explicit_season:
        chat_context["season"] = explicit_season

    state = chat_context.get("state")
    season = chat_context.get("season")

    # Structured retrieval is used when entities are known.
    if state and season:
        g = df[(df["State"] == state) & (df["Season"] == season)]
        if len(g):
            return f"In {state}, during {season}, the average potato modal price in the collected dataset is Rs {g['Modal_Price_Rs_per_Quintal'].mean():.0f} per quintal."

    if state:
        g = latest[latest["State"] == state]
        if len(g):
            return f"As of {latest_date.date()}, {state} averages Rs {g['Modal_Price_Rs_per_Quintal'].mean():.0f} per quintal across {len(g)} market records."

    if season:
        g = df[df["Season"] == season]
        if len(g):
            return f"Nationally, the average potato modal price during {season} is Rs {g['Modal_Price_Rs_per_Quintal'].mean():.0f} per quintal in the collected dataset."

    retrieved = retrieve(question, top_k=1)
    doc, score = retrieved[0]
    return f"Retrieved information: {doc['text']}"

demo = [
    "What is the current potato price in Tamil Nadu?",
    "And what about during monsoon there?",
    "What about winter there?"
]
for q in demo:
    print("User:", q)
    print("Bot :", rag_answer(q))
    print()

## 9. Model interpretation and dashboard exports

In [ ]:
feature_importance = pd.Series(
    best_model.feature_importances_, index=X_lag.columns
).sort_values(ascending=False).head(10)

print("Top features:")
display(feature_importance.to_frame("importance"))

dashboard_summary = {
    "dataset": {
        "records": int(len(df)),
        "states": int(df["State"].nunique()),
        "markets": int(df["Market"].nunique()),
        "date_start": str(df["Arrival_Date"].min().date()),
        "date_end": str(df["Arrival_Date"].max().date()),
        "note": "52-week history and weather fields are documented simulation anchored to a real market snapshot."
    },
    "model_comparison": results_df.round(4).to_dict(orient="index"),
    "hyperparameters": hp_df.round(4).to_dict(orient="records"),
    "best_params": grid.best_params_,
    "best_cv_r2": float(grid.best_score_),
    "test_metrics": {k: float(v) for k,v in tuned_metrics.items()},
    "season_average": season_avg.round(2).to_dict(),
    "top_features": feature_importance.round(6).to_dict()
}

with open(os.path.join(ROOT, "outputs", "dashboard_summary.json"), "w") as f:
    json.dump(dashboard_summary, f, indent=2)

print("Saved outputs/dashboard_summary.json")

## 10. Conclusion

- The project predicts weekly potato modal prices using seasonal, weather and historical-price features.
- Lag/rolling features materially improve the forecasting task in this dataset.
- Multiple regression models and Random Forest hyperparameters are evaluated using MAE, RMSE and R².
- The latest 8 weeks are kept as a chronological holdout for the main test evaluation.
- Six natural-language prompts are documented and used for data extraction/querying.
- The chatbot uses TF-IDF retrieval over data-derived documents and remembers state/season context across turns.
- The dashboard presents the model comparison, hyperparameter sensitivity, seasonal pattern and conversational interface.
- The major limitation is data provenance: the 52-week historical/weather fields are simulated. Real deployment requires genuine multi-year market history and observed weather data.